# Reglas de Asociación _(versión detallada)_

_Market Basket Analysis, Apriori, métricas (soporte, confianza, lift) y caso real_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

---
> 📖 **Nota:** Esta es la versión detallada del notebook `04_reglas_asociacion.ipynb`.  
> Incluye teoría ampliada y comentarios línea a línea en cada bloque de código.

![Aprendizaje No Supervisado](assets/header.png)

## 1. ¿Qué son las reglas de asociación?

Las reglas de asociación son patrones del tipo **"si compras A, entonces también compras B"** que se descubren automáticamente en grandes bases de datos transaccionales.

### 1.1 Definiciones fundamentales

**Transacción:** Conjunto de ítems comprados/consumidos en un evento.

**Ejemplo:**
```
T1 = {pan, leche, mantequilla}
T2 = {pan, huevos}
T3 = {leche, mantequilla, café}
T4 = {pan, leche, mantequilla, huevos}
```

**Itemset:** Conjunto de uno o más ítems.
- Itemset de tamaño 1: {pan}, {leche}
- Itemset de tamaño 2: {pan, leche}, {leche, mantequilla}
- Itemset de tamaño 3: {pan, leche, mantequilla}

**Regla de asociación:** Implicación de la forma $A \Rightarrow B$ donde:
- $A$: **antecedente** (itemset de entrada)
- $B$: **consecuente** (itemset de salida)
- $A \cap B = \emptyset$ (disjuntos — no se solapan)

**Ejemplo de reglas:**
- $\{\text{pan}, \text{leche}\} \Rightarrow \{\text{mantequilla}\}$
- $\{\text{café}\} \Rightarrow \{\text{azúcar}\}$

### 1.2 Diferencia con otros problemas de ML

| Aspecto | Clasificación | Clustering | Reglas de Asociación |
|---------|---------------|------------|----------------------|
| **Datos** | $(X, y)$ etiquetados | $X$ sin etiquetas | Transacciones (conjuntos) |
| **Objetivo** | Predecir $y$ | Agrupar observaciones | Encontrar co-ocurrencias |
| **Output** | Clase o valor | Asignación a clusters | Reglas $A \Rightarrow B$ |
| **Ejemplo** | ¿Este cliente hará churn? | ¿Qué segmentos hay? | ¿Qué productos se compran juntos? |

### 1.3 Casos de uso

**Retail y E-commerce:**
- Diseño de góndolas (productos que se compran juntos cerca)
- Promociones cruzadas ("lleva 2 y paga 1")
- Recomendaciones ("clientes que compraron X también compraron Y")

**Salud:**
- Síntomas que aparecen juntos (diagnóstico)
- Efectos secundarios de medicamentos
- Comorbilidades (enfermedades que co-ocurren)

**Web analytics:**
- Secuencias de clicks en un sitio
- Páginas que se visitan juntas
- Patrones de navegación

**Finanzas:**
- Productos financieros que se contratan juntos
- Patrones de fraude (transacciones sospechosas que co-ocurren)

**Telecomunicaciones:**
- Servicios que se contratan juntos (internet + streaming)
- Patrones de uso (llamadas + datos)

### 1.4 Limitaciones importantes

❌ **No implica causalidad:** $A \Rightarrow B$ NO significa que $A$ cause $B$
- Ejemplo: {pañales} → {cerveza} (famoso caso de Walmart)
- Correlación, no causalidad (padres jóvenes compran ambos)

❌ **Datos esparsos:** La mayoría de itemsets tienen soporte muy bajo
- Con 1000 productos, hay $2^{1000}$ itemsets posibles
- Solo una fracción minúscula aparece en las transacciones

❌ **Explosión combinatoria:** El número de reglas crece exponencialmente
- Necesitamos umbrales (min_support, min_confidence) para filtrar

❌ **Interpretabilidad:** Muchas reglas pueden ser obvias o irrelevantes
- Requiere validación de negocio

## 2. Métricas fundamentales

Para evaluar la calidad de una regla $A \Rightarrow B$, usamos tres métricas principales.

### 2.1 Soporte (Support)

**Definición:** Frecuencia con la que un itemset aparece en las transacciones.

**Fórmula para un itemset $X$:**

$$
\text{sop}(X) = \frac{|\{T \in D : X \subseteq T\}|}{|D|}
$$

donde:
- $D$: conjunto de todas las transacciones
- $|D|$: número total de transacciones
- $|\{T \in D : X \subseteq T\}|$: número de transacciones que contienen $X$

**Fórmula para una regla $A \Rightarrow B$:**

$$
\text{sop}(A \Rightarrow B) = \text{sop}(A \cup B) = P(A \cap B)
$$

**Interpretación:**
- Proporción de transacciones que contienen **tanto** $A$ **como** $B$
- Rango: $[0, 1]$ (0% a 100%)

**Ejemplo:**
```
Total de transacciones: 1000
Transacciones con {pan, leche}: 150

sop({pan, leche}) = 150/1000 = 0.15 = 15%
```

**Uso práctico:**
- Filtrar itemsets raros con `min_support` (e.g., 5%)
- Itemsets muy raros no son útiles para recomendaciones masivas
- Itemsets muy frecuentes pueden ser obvios (e.g., bolsas plásticas)

**Propiedades:**
- **Anti-monotonía:** Si $X$ es infrecuente, cualquier superconjunto de $X$ también lo es
  $$\text{sop}(X) < \text{min\_support} \implies \text{sop}(X \cup Y) < \text{min\_support}$$
- Esta propiedad es la base del algoritmo Apriori (poda eficiente)

---

### 2.2 Confianza (Confidence)

**Definición:** Probabilidad condicional de comprar $B$ dado que se compró $A$.

**Fórmula:**

$$
\text{conf}(A \Rightarrow B) = \frac{\text{sop}(A \cup B)}{\text{sop}(A)} = P(B | A)
$$

**Interpretación:**
- De las transacciones que contienen $A$, ¿qué proporción también contiene $B$?
- Rango: $[0, 1]$ (0% a 100%)

**Ejemplo:**
```
Transacciones con {pan, leche}: 150
Transacciones con {pan, leche, mantequilla}: 120

conf({pan, leche} → {mantequilla}) = 120/150 = 0.80 = 80%
```

**Interpretación del ejemplo:**
- Si un cliente compra pan y leche, hay 80% de probabilidad de que también compre mantequilla

**Limitación importante:**
- Alta confianza NO garantiza una regla útil
- Si $B$ es muy popular (e.g., 90% de transacciones), la confianza será alta aunque no haya asociación real

**Ejemplo del problema:**
```
sop({café}) = 0.10 (10% de transacciones)
sop({bolsas}) = 0.90 (90% de transacciones — casi todos compran bolsas)
sop({café, bolsas}) = 0.09 (9% de transacciones)

conf({café} → {bolsas}) = 0.09/0.10 = 0.90 = 90%
```

La confianza es alta (90%) pero la regla es inútil — las bolsas son populares independientemente del café.

**Solución:** Usar **lift** para detectar asociaciones reales.

---

### 2.3 Lift

**Definición:** Cuánto más probable es comprar $B$ junto con $A$ comparado con comprar $B$ independientemente.

**Fórmula:**

$$
\text{lift}(A \Rightarrow B) = \frac{\text{conf}(A \Rightarrow B)}{\text{sop}(B)} = \frac{\text{sop}(A \cup B)}{\text{sop}(A) \cdot \text{sop}(B)} = \frac{P(A \cap B)}{P(A) \cdot P(B)}
$$

**Interpretación:**

| Lift | Significado | Acción |
|------|-------------|--------|
| **> 1** | $A$ y $B$ se compran juntos **más** que por azar → asociación positiva | **Regla útil** — promoción cruzada |
| **= 1** | Independientes → comprar $A$ no afecta la probabilidad de comprar $B$ | Regla inútil — ignorar |
| **< 1** | Se evitan entre sí (productos sustitutos) → asociación negativa | Posible canibalización |

**Ejemplo 1 (asociación positiva):**
```
sop({pan, leche}) = 0.15
sop({pan}) = 0.30
sop({leche}) = 0.40

lift = 0.15 / (0.30 × 0.40) = 0.15 / 0.12 = 1.25
```

Interpretación: Comprar pan y leche juntos es **1.25× más probable** que si fueran independientes.

**Ejemplo 2 (independencia):**
```
sop({café, bolsas}) = 0.09
sop({café}) = 0.10
sop({bolsas}) = 0.90

lift = 0.09 / (0.10 × 0.90) = 0.09 / 0.09 = 1.0
```

Interpretación: Café y bolsas son **independientes** — no hay asociación real.

**Ejemplo 3 (asociación negativa):**
```
sop({Coca-Cola, Pepsi}) = 0.01
sop({Coca-Cola}) = 0.20
sop({Pepsi}) = 0.15

lift = 0.01 / (0.20 × 0.15) = 0.01 / 0.03 = 0.33
```

Interpretación: Comprar ambas es **0.33× menos probable** que por azar — son sustitutos.

**Uso práctico:**
- Filtrar reglas con `lift >= 1.2` (asociación fuerte)
- Ordenar reglas por lift descendente
- Detectar productos sustitutos (lift < 1)

**Propiedades:**
- **Simétrico:** $\text{lift}(A \Rightarrow B) = \text{lift}(B \Rightarrow A)$
- **No acotado superiormente:** puede ser arbitrariamente grande
- **Sensible a itemsets raros:** lift puede ser muy alto para itemsets con soporte muy bajo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ejemplo visual: Diagrama de Venn para entender las métricas

# Simulamos 1000 transacciones
n = 1000

# Definimos las frecuencias
n_A = 300        # 300 transacciones con A
n_B = 400        # 400 transacciones con B
n_AB = 150       # 150 transacciones con A y B

# Calculamos las métricas
sop_A = n_A / n
sop_B = n_B / n
sop_AB = n_AB / n
conf_A_B = n_AB / n_A
lift_A_B = sop_AB / (sop_A * sop_B)

# --- Visualización con diagrama de Venn ---
from matplotlib.patches import Circle
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 6))

# Círculo A (izquierda)
circle_A = Circle((0.35, 0.5), 0.25, color='blue', alpha=0.3, label='A')
ax.add_patch(circle_A)

# Círculo B (derecha)
circle_B = Circle((0.65, 0.5), 0.25, color='red', alpha=0.3, label='B')
ax.add_patch(circle_B)

# Anotaciones
ax.text(0.25, 0.5, f'Solo A\n{n_A - n_AB}', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(0.75, 0.5, f'Solo B\n{n_B - n_AB}', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(0.5, 0.5, f'A ∩ B\n{n_AB}', ha='center', va='center', fontsize=12, fontweight='bold', color='purple')
ax.text(0.5, 0.15, f'Ni A ni B\n{n - n_A - n_B + n_AB}', ha='center', va='center', fontsize=10)

# Título y métricas
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Diagrama de Venn — Regla: A → B\n(Total de transacciones: {n})', fontsize=13, fontweight='bold')

# Leyenda con las métricas
metricas_text = f"""
Métricas de la regla A → B:

• Soporte(A ∪ B) = {n_AB}/{n} = {sop_AB:.2f} ({sop_AB*100:.0f}%)
  → {sop_AB*100:.0f}% de transacciones tienen A y B

• Confianza(A → B) = {n_AB}/{n_A} = {conf_A_B:.2f} ({conf_A_B*100:.0f}%)
  → Si compras A, hay {conf_A_B*100:.0f}% de probabilidad de comprar B

• Lift(A → B) = {sop_AB:.2f} / ({sop_A:.2f} × {sop_B:.2f}) = {lift_A_B:.2f}
  → Comprar A y B juntos es {lift_A_B:.2f}× más probable que por azar
"""

ax.text(0.5, -0.15, metricas_text, ha='center', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
        family='monospace')

plt.tight_layout()
plt.show()

# INTERPRETACIÓN:
# - Soporte: mide la FRECUENCIA de la regla (qué tan común es)
# - Confianza: mide la PRECISIÓN de la regla (qué tan confiable es)
# - Lift: mide la FUERZA de la asociación (qué tan útil es)
#
# REGLA DE ORO:
# Una buena regla debe tener:
# 1. Soporte razonable (no muy raro, e.g., ≥ 5%)
# 2. Confianza alta (e.g., ≥ 40-60%)
# 3. Lift > 1 (idealmente ≥ 1.2)

## 3. Algoritmo Apriori

Apriori es el algoritmo clásico para encontrar itemsets frecuentes y generar reglas de asociación. Fue propuesto por Agrawal & Srikant (1994).

### 3.1 Idea central: Propiedad anti-monótona

> **Propiedad anti-monótona del soporte:**  
> Si un itemset $X$ es infrecuente (soporte < `min_support`), entonces **cualquier superconjunto** de $X$ también es infrecuente.

**Formulación matemática:**

$$
\text{sop}(X) < \text{min\_support} \implies \forall Y \supseteq X: \text{sop}(Y) < \text{min\_support}
$$

**Intuición:** Si {pan, leche} aparece en solo 2% de transacciones, entonces {pan, leche, mantequilla} aparecerá en ≤ 2% (no puede ser más frecuente).

**Implicación:** Podemos **podar** el espacio de búsqueda sin calcular el soporte de todos los itemsets.

### 3.2 Algoritmo paso a paso

**Entrada:**
- $D$: base de datos de transacciones
- `min_support`: umbral de soporte mínimo (e.g., 0.05 = 5%)

**Salida:**
- $L$: conjunto de todos los itemsets frecuentes

**Pasos:**

#### Paso 1: Encontrar itemsets frecuentes de tamaño 1

```
L₁ = {itemsets de tamaño 1 con soporte ≥ min_support}
```

**Ejemplo:**
```
min_support = 0.05 (5%)
Total transacciones: 1000

Conteo:
{pan}: 300 transacciones → sop = 0.30 ✓ (frecuente)
{leche}: 250 transacciones → sop = 0.25 ✓ (frecuente)
{café}: 30 transacciones → sop = 0.03 ✗ (infrecuente, descartado)

L₁ = {{pan}, {leche}}
```

#### Paso 2: Generar candidatos de tamaño 2

```
C₂ = {combinaciones de itemsets en L₁}
```

**Ejemplo:**
```
L₁ = {{pan}, {leche}}
C₂ = {{pan, leche}}
```

#### Paso 3: Filtrar candidatos por soporte

```
L₂ = {itemsets en C₂ con soporte ≥ min_support}
```

**Ejemplo:**
```
{pan, leche}: 150 transacciones → sop = 0.15 ✓ (frecuente)
L₂ = {{pan, leche}}
```

#### Paso 4: Repetir para tamaños 3, 4, ...

```
Para k = 3, 4, 5, ...:
  1. Generar Cₖ combinando itemsets de Lₖ₋₁
  2. Filtrar Lₖ = {itemsets en Cₖ con soporte ≥ min_support}
  3. Si Lₖ está vacío, DETENER
```

**Resultado final:**

$$
L = L_1 \cup L_2 \cup L_3 \cup \dots
$$

### 3.3 Generación de reglas

Una vez tenemos los itemsets frecuentes, generamos reglas:

**Para cada itemset frecuente $I \in L$ con $|I| \ge 2$:**

1. Generar todas las particiones $A, B$ donde $A \cup B = I$ y $A \cap B = \emptyset$
2. Para cada partición, crear la regla $A \Rightarrow B$
3. Calcular confianza y lift
4. Filtrar reglas con `conf ≥ min_confidence` y `lift ≥ min_lift`

**Ejemplo:**
```
Itemset frecuente: {pan, leche, mantequilla}

Reglas posibles:
{pan} → {leche, mantequilla}
{leche} → {pan, mantequilla}
{mantequilla} → {pan, leche}
{pan, leche} → {mantequilla}
{pan, mantequilla} → {leche}
{leche, mantequilla} → {pan}
```

### 3.4 Complejidad y optimizaciones

**Complejidad:**
- **Tiempo:** O($2^{|I|}$) en el peor caso (exponencial en el número de ítems)
- **Espacio:** O($|L|$) (almacenar itemsets frecuentes)

**Optimizaciones:**
- **Poda por anti-monotonía:** Evita calcular soporte de superconjuntos de itemsets infrecuentes
- **Hash trees:** Estructura de datos para contar soporte eficientemente
- **Particionamiento:** Dividir la base de datos en particiones y procesar en paralelo

**Limitaciones:**
- ❌ Genera muchos candidatos (costoso en memoria)
- ❌ Múltiples pasadas sobre los datos (I/O intensivo)
- ❌ No escala bien a millones de transacciones

**Alternativa más eficiente:** **FP-Growth** (Frequent Pattern Growth)
- Solo 2 pasadas sobre los datos
- No genera candidatos explícitos
- Usa un árbol comprimido (FP-tree)
- 1-2 órdenes de magnitud más rápido que Apriori

## 5. Caso práctico — canasta de mercado sintética

Para mantener el notebook autocontenido generamos un conjunto pequeño de transacciones de supermercado. En un caso real usarías el log transaccional de tu negocio o un dataset abierto como **Online Retail (UCI)** — la mecánica es exactamente la misma.

> ⚠️ Este notebook usa `mlxtend` (`uv add mlxtend` o ya incluido en `pyproject.toml`).

In [ ]:
from pathlib import Path
import pandas as pd

# Definimos la ruta al CSV
DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Verificamos que el archivo existe
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

# Cargamos el CSV completo
df = pd.read_csv(DATA)

# Limpieza: TotalCharges viene como string con espacios
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)

print('Shape:', df.shape)
print(f'Clientes después de limpiar: {len(df)}')

# Mostramos las primeras filas
df.head()

# CONTEXTO DEL DATASET:
# - 7032 clientes de una compañía de telecomunicaciones
# - Variables demográficas: gender, SeniorCitizen, Partner, Dependents
# - Servicios contratados: PhoneService, InternetService, StreamingTV,
#   StreamingMovies, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport
# - Contrato: Contract, PaymentMethod, PaperlessBilling
# - Facturación: MonthlyCharges, TotalCharges, tenure
# - Target: Churn (si se dio de baja)
#
# OBJETIVO:
# - Descubrir qué servicios se contratan juntos
# - Generar reglas del tipo: "Si contrata InternetService=Fiber, también contrata StreamingTV"
# - Usar estas reglas para:
#   * Cross-selling (ofrecer servicios complementarios)
#   * Bundling (crear paquetes de servicios)
#   * Diseño de planes (servicios que van bien juntos)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Selección de servicios (ítems)
# ─────────────────────────────────────────────────────────────────
# Nos enfocamos en los servicios contratados (variables categóricas binarias)

servicios = [
    'PhoneService',       # Servicio telefónico (Yes/No)
    'InternetService',    # Servicio de internet (DSL/Fiber optic/No)
    'OnlineSecurity',     # Seguridad online (Yes/No/No internet service)
    'OnlineBackup',       # Backup online (Yes/No/No internet service)
    'DeviceProtection',   # Protección de dispositivos (Yes/No/No internet service)
    'TechSupport',        # Soporte técnico (Yes/No/No internet service)
    'StreamingTV',        # TV en streaming (Yes/No/No internet service)
    'StreamingMovies',    # Películas en streaming (Yes/No/No internet service)
]

# Extraemos solo las columnas de servicios
df_servicios = df[servicios].copy()

print('Servicios seleccionados:')
print(df_servicios.head())
print(f'\nShape: {df_servicios.shape}')

# OBSERVACIÓN:
# - Cada fila es un "cliente" (equivalente a una "transacción" en retail)
# - Cada columna es un "servicio" (equivalente a un "producto")
# - Los valores son categóricos (Yes/No/DSL/Fiber optic/No internet service)
#
# PROBLEMA:
# - Apriori requiere datos en formato "basket" (matriz binaria)
# - Necesitamos convertir las categorías a columnas binarias (one-hot encoding)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Conversión a formato "basket" (one-hot encoding)
# ─────────────────────────────────────────────────────────────────

# get_dummies convierte cada categoría en una columna binaria
# drop_first=False: conservamos TODAS las categorías (no descartamos la primera)
# Esto es importante para reglas de asociación — queremos ver todas las combinaciones
basket = pd.get_dummies(df_servicios, drop_first=False)

# Convertimos a booleano (True/False) para mlxtend
# mlxtend requiere valores booleanos, no 0/1
basket = basket.astype(bool)

print('Formato basket (primeras 5 filas):')
print(basket.head())
print(f'\nShape: {basket.shape}')
print(f'Columnas (ítems): {basket.shape[1]}')

# RESULTADO:
# - Cada fila es un cliente (transacción)
# - Cada columna es un servicio específico (ítem)
# - True = el cliente tiene ese servicio
# - False = el cliente NO tiene ese servicio
#
# EJEMPLO DE INTERPRETACIÓN:
# Fila 0:
# - PhoneService_Yes = True → tiene servicio telefónico
# - InternetService_DSL = True → tiene internet DSL
# - OnlineSecurity_No = True → NO tiene seguridad online
# - StreamingTV_No = True → NO tiene streaming TV
#
# NOTA: Algunas columnas son mutuamente excluyentes:
# - InternetService_DSL, InternetService_Fiber optic, InternetService_No
#   (un cliente solo puede tener uno de estos)
# - Esto es OK para reglas de asociación — queremos ver qué combinaciones ocurren


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import seaborn as sns

# ─────────────────────────────────────────────────────────────────
# PASO 1: Encontrar itemsets frecuentes con Apriori
# ─────────────────────────────────────────────────────────────────

# apriori encuentra todos los itemsets con soporte ≥ min_support
# min_support=0.10: solo itemsets que aparecen en ≥ 10% de clientes
# use_colnames=True: usa nombres de columnas en lugar de índices numéricos
itemsets = apriori(basket, min_support=0.10, use_colnames=True)

# Ordenamos por soporte descendente (más frecuentes primero)
itemsets = itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f'Itemsets frecuentes encontrados: {len(itemsets)}')
print('\nTop 10 itemsets más frecuentes:')
print(itemsets.head(10))

# INTERPRETACIÓN DE LA TABLA:
# - support: proporción de clientes que tienen ese itemset
# - itemsets: conjunto de servicios (frozenset)
#
# EJEMPLO:
# Si support = 0.45 para {PhoneService_Yes}
# → 45% de los clientes tienen servicio telefónico
#
# Si support = 0.25 para {InternetService_Fiber optic, StreamingTV_Yes}
# → 25% de los clientes tienen fibra óptica Y streaming TV
#
# OBSERVACIONES ESPERADAS:
# - Itemsets de tamaño 1 (individuales) tienen mayor soporte
# - Itemsets de tamaño 2-3 tienen menor soporte (menos comunes)
# - Itemsets muy grandes (4+) son raros (soporte bajo)
#
# NOTA: Con min_support=0.10, filtramos itemsets que aparecen en < 10% de clientes
# Esto reduce el número de itemsets de millones a cientos/miles


In [ ]:
# ─────────────────────────────────────────────────────────────────
# PASO 2: Generar reglas de asociación
# ─────────────────────────────────────────────────────────────────

# association_rules genera todas las reglas posibles a partir de los itemsets
# metric='confidence': filtra por confianza mínima
# min_threshold=0.40: solo reglas con confianza ≥ 40%
reglas = association_rules(itemsets, metric='confidence', min_threshold=0.40)

# Filtramos adicionalmente por lift ≥ 1.0 (asociación real, no por azar)
reglas = reglas[reglas['lift'] >= 1.0]

# Ordenamos por lift descendente (asociaciones más fuertes primero)
reglas = reglas.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Reglas encontradas: {len(reglas)}')
print('\nColumnas de la tabla de reglas:')
print(reglas.columns.tolist())

# COLUMNAS DE LA TABLA:
# - antecedents: itemset de entrada (A)
# - consequents: itemset de salida (B)
# - antecedent support: soporte de A
# - consequent support: soporte de B
# - support: soporte de A ∪ B (la regla completa)
# - confidence: P(B|A) = support / antecedent support
# - lift: support / (antecedent support × consequent support)
# - leverage: support - (antecedent support × consequent support)
# - conviction: (1 - consequent support) / (1 - confidence)
#
# MÉTRICAS ADICIONALES (menos usadas):
# - leverage: diferencia entre soporte observado y esperado (si fueran independientes)
# - conviction: qué tan dependiente es B de A (similar a lift pero asimétrico)
#
# FILTROS APLICADOS:
# 1. min_support=0.10 en apriori → itemsets frecuentes
# 2. min_confidence=0.40 en association_rules → reglas confiables
# 3. lift ≥ 1.0 → asociaciones reales (no por azar)
#
# RESULTADO:
# Solo reglas que son:
# - Frecuentes (≥ 10% de clientes)
# - Confiables (≥ 40% de probabilidad)
# - Útiles (lift ≥ 1.0)


import matplotlib.pyplot as plt

# Convertimos frozensets a strings para mejor visualización
# frozenset es un tipo de dato inmutable (como set pero hasheable)
# Lo convertimos a string para mostrarlo en tablas y gráficos
reglas['antecedents_str'] = reglas['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas['consequents_str'] = reglas['consequents'].apply(lambda x: ', '.join(list(x)))

# Mostramos las top 15 reglas por lift
print('Top 15 reglas por lift (asociaciones más fuertes):')
print(reglas[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]
      .head(15)
      .round(3)
      .to_string(index=False))

# --- Visualización: Scatter plot de confianza vs lift ---
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot con tamaño proporcional al soporte
# s=reglas['support']*1000: tamaño del punto proporcional al soporte
# alpha=0.6: transparencia para ver superposiciones
# c=reglas['lift']: color según lift (gradiente)
scatter = ax.scatter(reglas['confidence'], reglas['lift'],
                     s=reglas['support']*1000, alpha=0.6,
                     c=reglas['lift'], cmap='viridis', edgecolor='black')

# Líneas de referencia
ax.axhline(1.0, color='red', linestyle='--', linewidth=2, label='Lift = 1 (independencia)')
ax.axvline(0.5, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Conf = 50%')

ax.set_xlabel('Confianza (Confidence)', fontsize=12)
ax.set_ylabel('Lift', fontsize=12)
ax.set_title('Reglas de Asociación: Confianza vs Lift\n(Tamaño del punto = soporte)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Colorbar para interpretar el lift
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Lift', fontsize=11)

plt.tight_layout()
plt.show()

# CÓMO INTERPRETAR EL GRÁFICO:
#
# EJE X (Confianza):
# - Izquierda (baja): reglas poco confiables (< 50%)
# - Derecha (alta): reglas muy confiables (> 80%)
#
# EJE Y (Lift):
# - Abajo (lift ≈ 1): asociación débil (casi independientes)
# - Arriba (lift > 1.5): asociación fuerte
#
# TAMAÑO DEL PUNTO (Soporte):
# - Puntos grandes: reglas frecuentes (muchos clientes)
# - Puntos pequeños: reglas raras (pocos clientes)
#
# COLOR (Lift):
# - Amarillo: lift alto (asociación fuerte)
# - Morado: lift bajo (asociación débil)
#
# REGLAS IDEALES (esquina superior derecha):
# - Alta confianza (> 60%)
# - Alto lift (> 1.2)
# - Soporte razonable (punto mediano/grande)
#
# REGLAS A EVITAR:
# - Lift < 1: asociación negativa (productos sustitutos)
# - Confianza baja (< 40%): poco confiables
# - Soporte muy bajo (puntos diminutos): demasiado raras

In [ ]:
# --- Visualización: scatter support vs confidence, color = lift ---
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    reglas['support'], reglas['confidence'],
    c=reglas['lift'], cmap='viridis', s=60, edgecolor='white',
)
ax.set_xlabel('Support'); ax.set_ylabel('Confidence')
ax.set_title('Reglas — color = lift (más oscuro = más fuerte)')
plt.colorbar(sc, ax=ax, label='Lift')
plt.show()


In [ ]:
## 5. Interpretación de negocio

Las reglas de asociación son **herramientas de exploración**, no de predicción. Su valor está en revelar patrones ocultos que pueden traducirse en acciones de negocio.

### 5.1 Cómo interpretar una regla

**Ejemplo de regla:**
```
{InternetService_Fiber optic} → {StreamingTV_Yes}
support = 0.25, confidence = 0.70, lift = 1.8
```

**Interpretación:**
- **Soporte (0.25):** 25% de los clientes tienen fibra óptica Y streaming TV
  - Es una combinación **común** (1 de cada 4 clientes)
  
- **Confianza (0.70):** De los clientes con fibra óptica, 70% tienen streaming TV
  - Si un cliente contrata fibra, hay **70% de probabilidad** de que también contrate streaming TV
  
- **Lift (1.8):** Tener fibra óptica hace 1.8× más probable contratar streaming TV
  - La asociación es **fuerte** (no es por azar)
  - Si fueran independientes, solo 39% de clientes con fibra tendrían streaming TV (70% / 1.8)

### 5.2 Acciones de negocio por tipo de regla

#### Tipo 1: Cross-selling (venta cruzada)

**Regla:** {Servicio A} → {Servicio B} con alta confianza y lift

**Acción:**
- Cuando un cliente contrata A, **ofrecerle B** automáticamente
- Descuento por contratar A + B juntos
- Campaña de email: "Clientes con A también disfrutan de B"

**Ejemplo:**
```
{InternetService_Fiber optic} → {StreamingMovies_Yes}
```
→ Ofrecer streaming de películas a clientes con fibra óptica

---

#### Tipo 2: Bundling (paquetes)

**Regla:** Múltiples servicios con alto soporte y lift

**Acción:**
- Crear **paquetes** de servicios que se contratan juntos
- Precio especial por el paquete (más barato que por separado)
- Simplificar la oferta (menos opciones, más claras)

**Ejemplo:**
```
{InternetService_Fiber optic, StreamingTV_Yes, StreamingMovies_Yes}
support = 0.20
```
→ Paquete "Entretenimiento Premium": Fibra + TV + Películas

---

#### Tipo 3: Upselling (venta superior)

**Regla:** {Servicio básico} → {Servicio premium} con lift > 1

**Acción:**
- Ofrecer upgrade de servicio básico a premium
- Mostrar beneficios del servicio premium
- Trial gratuito del servicio premium

**Ejemplo:**
```
{InternetService_DSL} → {InternetService_Fiber optic}
```
→ Campaña de upgrade de DSL a fibra óptica

---

#### Tipo 4: Retención (churn prevention)

**Regla:** {Servicios} → {Churn_Yes} con alta confianza

**Acción:**
- Identificar combinaciones de servicios con alto riesgo de churn
- Ofrecer descuentos o mejoras para retener al cliente
- Contacto proactivo del equipo de retención

**Ejemplo:**
```
{Contract_Month-to-month, InternetService_No} → {Churn_Yes}
```
→ Clientes sin internet y sin contrato tienen alto riesgo de irse

---

### 5.3 Validación de reglas

No todas las reglas con buenas métricas son útiles. Debes validar:

#### 1. ¿Es accionable?

**Pregunta:** ¿Podemos hacer algo con esta regla?

**Ejemplo de regla NO accionable:**
```
{gender_Female} → {Partner_Yes}
```
→ No podemos cambiar el género o estado civil del cliente

**Ejemplo de regla accionable:**
```
{PhoneService_Yes} → {InternetService_Fiber optic}
```
→ Podemos ofrecer internet a clientes con teléfono

---

#### 2. ¿Es obvia?

**Pregunta:** ¿Es una regla que ya sabíamos?

**Ejemplo de regla obvia:**
```
{InternetService_No} → {OnlineSecurity_No internet service}
```
→ Obvio — sin internet no puede tener servicios online

**Ejemplo de regla interesante:**
```
{TechSupport_Yes} → {Churn_No}
```
→ Insight: el soporte técnico reduce el churn (no era obvio)

---

#### 3. ¿Es rentable?

**Pregunta:** ¿El costo de implementar la acción es menor que el beneficio?

**Cálculo:**
```
Beneficio esperado = (# clientes afectados) × (tasa de conversión) × (valor del servicio)
Costo = (# clientes contactados) × (costo de campaña por cliente)

ROI = (Beneficio - Costo) / Costo
```

**Ejemplo:**
```
Regla: {PhoneService_Yes} → {InternetService_Fiber optic}
Clientes con solo teléfono: 1000
Tasa de conversión esperada (confidence): 30%
Valor mensual de fibra: $50
Costo de campaña por cliente: $2

Beneficio = 1000 × 0.30 × $50 = $15,000/mes
Costo = 1000 × $2 = $2,000
ROI = ($15,000 - $2,000) / $2,000 = 650%
```
→ Regla muy rentable

---

### 5.4 Limitaciones y advertencias

❌ **No implica causalidad:** A → B no significa que A cause B

**Ejemplo:**
```
{SeniorCitizen_1} → {Contract_Month-to-month}
```
→ Correlación, no causalidad (adultos mayores prefieren flexibilidad)

❌ **Puede haber variables confusoras:** Una tercera variable puede explicar la asociación

**Ejemplo:**
```
{StreamingTV_Yes} → {StreamingMovies_Yes}
```
→ Ambos requieren internet de alta velocidad (variable confusora)

❌ **Datos históricos:** Las reglas reflejan el pasado, no el futuro

**Ejemplo:**
```
Regla descubierta en 2020: {PhoneService_Yes} → {InternetService_DSL}
```
→ En 2026, DSL está obsoleto (fibra es el estándar)

❌ **Sesgo de selección:** Solo vemos clientes que ya contrataron

**Ejemplo:**
```
No vemos: {InternetService_Fiber optic} → {Churn por precio alto}
```
→ Clientes que no contrataron fibra por precio alto no están en los datos


In [ ]:
print('Recomendaciones para quien compra "cafe":')
recomendar(reglas, 'cafe')


## 6. Referencias y recursos adicionales

### Papers clásicos
- Agrawal, R. & Srikant, R. (1994). *Fast Algorithms for Mining Association Rules*. VLDB.
- Han, J., Pei, J. & Yin, Y. (2000). *Mining Frequent Patterns without Candidate Generation* (FP-Growth). SIGMOD.

### Libros
- Tan, P., Steinbach, M. & Kumar, V. (2005). *Introduction to Data Mining*, Cap. 6.
- Aggarwal, C. C. (2015). *Data Mining: The Textbook*, Cap. 4.

### Documentación
- mlxtend: https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/
- Efficient-Apriori (alternativa): https://github.com/tommyod/Efficient-Apriori

### Datasets públicos
- Online Retail (UCI): https://archive.ics.uci.edu/dataset/352/online+retail
- Instacart Market Basket: https://www.kaggle.com/c/instacart-market-basket-analysis
- Groceries dataset (R): https://www.kaggle.com/datasets/heeraldedhia/groceries-dataset

---

## Resumen ejecutivo

| Concepto | Qué mide | Rango | Mejor valor | Uso |
|----------|----------|-------|-------------|-----|
| **Soporte** | Frecuencia del itemset | [0, 1] | Depende | Filtrar itemsets raros |
| **Confianza** | P(B\|A) | [0, 1] | Alto | Medir precisión de la regla |
| **Lift** | Asociación real vs azar | [0, ∞) | > 1 | Detectar asociaciones útiles |

**Workflow típico:**
1. Preprocesar datos a formato basket (one-hot encoding)
2. Aplicar Apriori con `min_support` (e.g., 5-10%)
3. Generar reglas con `min_confidence` (e.g., 40-60%)
4. Filtrar por `lift >= 1.2`
5. Ordenar por lift descendente
6. Validar con negocio (¿es accionable? ¿es obvia? ¿es rentable?)
7. Implementar acciones (cross-selling, bundling, etc.)

**Limitaciones clave:**
- ❌ No implica causalidad
- ❌ Datos esparsos (explosión combinatoria)
- ❌ Requiere validación de negocio
- ❌ No escala bien a millones de ítems (usar FP-Growth)

## 7. FP-Growth — alternativa más rápida

Cuando el catálogo crece (miles de ítems), Apriori se vuelve lento por la generación de candidatos. **FP-Growth** comprime las transacciones en un árbol (FP-tree) y extrae itemsets frecuentes sin generar candidatos explícitos. La API en `mlxtend` es idéntica:

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

itemsets_fp = fpgrowth(basket, min_support=0.05, use_colnames=True)
print(f'FP-Growth devolvió {len(itemsets_fp)} itemsets — los mismos que Apriori (más rápido).')


## 8. Limitaciones que vale la pena conocer

- **No infiere causalidad**: que dos productos se compren juntos no significa que uno cause al otro. Pueden tener una causa común (estación, promoción, vecindario).
- **Reglas obvias**: muchas veces se filtra "todo el mundo compra bolsas" — hay que limpiar el catálogo o subir el lift.
- **Datos esparsos**: con pocos ítems repetidos por canasta, el soporte cae mucho.
- **Tiempo**: el orden temporal se pierde. Si te importa _qué se compra después de qué_, mira **sequential pattern mining** (PrefixSpan, SPADE).

## 9. Referencias

- Agrawal, R. & Srikant, R. (1994). *Fast Algorithms for Mining Association Rules*.
- Han, J., Pei, J. & Yin, Y. (2000). *Mining Frequent Patterns without Candidate Generation* (FP-Growth).
- mlxtend — Frequent Patterns: https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/
- Online Retail dataset (real, para profundizar): https://archive.ics.uci.edu/dataset/352/online+retail
- Aggarwal, C. C. (2014). *Frequent Pattern Mining*.

## Predicción sobre datos nuevos — uso del modelo en producción

A diferencia de un modelo supervisado, en reglas de asociación **no persistimos un "modelo"** sino la **tabla de reglas**. La inferencia es una búsqueda: dada la canasta actual del usuario, devolver las reglas cuyo antecedente coincide y ordenarlas por lift.

In [ ]:
# 1. Persistir las reglas como tabla (CSV / parquet) para el servicio de recomendación
reglas_export = reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
reglas_export.to_csv('reglas_asociacion.csv', index=False)
print(f'Guardadas {len(reglas_export)} reglas en reglas_asociacion.csv')


In [ ]:
# 2. Función de recomendación en línea
def recomendar_canasta(canasta_actual, reglas_df, top=5, min_lift=1.0):
    """Recomienda productos dado el conjunto actual del usuario."""
    canasta_actual = set(canasta_actual)
    sugerencias = []
    for _, r in reglas_df.iterrows():
        antecedente = set(r['antecedents'].split(', '))
        consecuente = set(r['consequents'].split(', '))
        # antecedente cubierto y consecuente aún no en canasta
        if antecedente.issubset(canasta_actual) and not consecuente & canasta_actual:
            if r['lift'] >= min_lift:
                sugerencias.append({
                    'sugerencia': ', '.join(sorted(consecuente)),
                    'lift':       round(r['lift'], 2),
                    'confidence': round(r['confidence'], 2),
                    'support':    round(r['support'], 3),
                })
    return (
        pd.DataFrame(sugerencias)
        .sort_values('lift', ascending=False)
        .drop_duplicates('sugerencia')
        .head(top)
        .reset_index(drop=True)
    )

print('Carrito actual: pan + leche')
recomendar_canasta(['pan', 'leche'], reglas)


In [ ]:
print('Carrito actual: pasta')
recomendar_canasta(['pasta'], reglas)


**Lecciones para producción:**
- Reentrenar las reglas cada cierto tiempo: el catálogo cambia, las modas cambian.
- Combinar las reglas con un **filtrado por inventario** (no recomendar lo que está agotado) y por **margen** (priorizar productos rentables).
- Para sistemas a gran escala (millones de transacciones, miles de ítems), pasar de Apriori a **FP-Growth** o frameworks distribuidos (`pyspark.ml.fpm.FPGrowth`).
- Evaluar la calidad **A/B testando** las recomendaciones en producción — el lift offline no garantiza un mejor _click-through_.